In [1]:
import os

In [2]:
%pwd

'd:\\NLP Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\NLP Project'

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [ ]:
from NexText.constants import *


In [8]:
from NexText.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from NexText.utils.common import read_yaml, create_directories
from NexText.entity.config_entity import ModelEvaluationConfig


class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:

        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            model_path=Path(config.model_path),
            tokenizer_path=Path(config.tokenizer_path),
            metric_file_name=Path(config.metric_file_name)
        )

        return model_evaluation_config

In [9]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_from_disk
import torch
import pandas as pd
from tqdm import tqdm

d:\NLP Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
import os

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_from_disk

import torch
import pandas as pd
from tqdm import tqdm

from rouge_score import rouge_scorer

from NexText.entity.config_entity import ModelEvaluationConfig


class ModelEvaluation:

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """
        Split the dataset into smaller batches.
        """
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i:i + batch_size]

    def calculate_metric_on_test_ds(
        self,
        dataset,
        model,
        tokenizer,
        batch_size=1,
        device=None,
        column_text="dialogue",
        column_summary="summary"
    ):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"

        article_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_text],
                batch_size
            )
        )

        target_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_summary],
                batch_size
            )
        )

        predictions = []
        references = []

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(article_batches),
            desc="Evaluating"
        ):

            inputs = tokenizer(
                article_batch,
                max_length=512,
                truncation=True,
                padding=True,
                return_tensors="pt"
            )

            inputs = {
                key: value.to(device)
                for key, value in inputs.items()
            }

            with torch.no_grad():

                summaries = model.generate(
                    **inputs,
                    length_penalty=0.8,
                    num_beams=4,
                    max_new_tokens=128
                )

            decoded_summaries = tokenizer.batch_decode(
                summaries,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )

            predictions.extend(decoded_summaries)
            references.extend(target_batch)

        # Calculate ROUGE scores
        scorer = rouge_scorer.RougeScorer(
            ["rouge1", "rouge2", "rougeL"],
            use_stemmer=True
        )

        rouge1_scores = []
        rouge2_scores = []
        rougeL_scores = []

        for prediction, reference in zip(
            predictions,
            references
        ):

            scores = scorer.score(
                reference,
                prediction
            )

            rouge1_scores.append(
                scores["rouge1"].fmeasure
            )

            rouge2_scores.append(
                scores["rouge2"].fmeasure
            )

            rougeL_scores.append(
                scores["rougeL"].fmeasure
            )

        rouge_dict = {
            "rouge1": sum(rouge1_scores) / len(rouge1_scores),
            "rouge2": sum(rouge2_scores) / len(rouge2_scores),
            "rougeL": sum(rougeL_scores) / len(rougeL_scores)
        }

        return rouge_dict

    def evaluate(self):

        device = (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

        print(f"Using device: {device}")

        tokenizer = AutoTokenizer.from_pretrained(
            self.config.tokenizer_path
        )

        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_path
        )

        model_pegasus = model_pegasus.to(device)

        model_pegasus.eval()

        # Load transformed dataset
        dataset_samsum_pt = load_from_disk(
            self.config.data_path
        )

        # Evaluate on test dataset
        test_dataset = dataset_samsum_pt["test"]

        score = self.calculate_metric_on_test_ds(
            dataset=test_dataset,
            model=model_pegasus,
            tokenizer=tokenizer,
            batch_size=1,
            device=device,
            column_text="dialogue",
            column_summary="summary"
        )

        print("\nEvaluation Results:")
        print("-------------------")

        for metric, value in score.items():
            print(f"{metric}: {value:.4f}")

        # Convert metrics to DataFrame
        df = pd.DataFrame(
            [score],
            index=["pegasus"]
        )

        # Make sure directory exists
        os.makedirs(
            os.path.dirname(
                str(self.config.metric_file_name)
            ),
            exist_ok=True
        )

        # Save metrics
        df.to_csv(
            self.config.metric_file_name
        )

        print(
            f"\nMetrics saved to: "
            f"{self.config.metric_file_name}"
        )

        return score

In [11]:
try:
    config = ConfigurationManager()

    model_evaluation_config = (
        config.get_model_evaluation_config()
    )

    model_evaluation = ModelEvaluation(
        config=model_evaluation_config
    )

    model_evaluation.evaluate()

except Exception as e:
    raise e

[ 2026-09-23 21:33:56,994 ] 21 NexText_logger - INFO - YAML file: D:\NLP Project\config\config.yaml loaded successfully.
[ 2026-09-23 21:33:57,004 ] 21 NexText_logger - INFO - YAML file: D:\NLP Project\params.yaml loaded successfully.
[ 2026-09-23 21:33:57,008 ] 49 NexText_logger - INFO - Created directory at: artifacts
[ 2026-09-23 21:33:57,010 ] 49 NexText_logger - INFO - Created directory at: artifacts/model_evaluation
Using device: cuda


Evaluating: 100%|██████████| 819/819 [13:10<00:00,  1.04it/s]

[ 2026-09-23 21:47:25,328 ] 83 absl - INFO - Using default tokenizer.



Evaluation Results:
-------------------
rouge1: 0.4520
rouge2: 0.2170
rougeL: 0.3566

Metrics saved to: artifacts\model_evaluation\metrics.csv
